# Food-101 compute and exact-recovery definitions

**Authoring status: NOT RUN.** This notebook contains reusable definition cells only. Importing its `ahr-export` cells does not benchmark, download data or train. The user requested creation without Run All; only notebook-format and Python-syntax checks are performed during authoring.

Purpose: support Notebook 02 with a matched 64/256/1,024-token synthetic compute sweep and an actual Food-101 configuration checkpoint-resume preflight. Inputs: registered configuration, project root, run artifact directory and (for preflight) prepared real-image data bundle and protocol. Outputs: hash-validated case JSON records, `compute_benchmark.json`, `preflight_certificate.json`, initial controls and recoverable smoke checkpoints.

Expected runtime once explicitly enabled: compute sweep minutes to tens of minutes depending on hardware; each case is bounded by checks between complete calls. Actual-config recovery runs eight updates per architecture across reference/split trajectories, expected minutes on the pilot accelerator but unmeasured until executed. Full training is never launched here.

Checkpoints: `checkpoints/<run_id>/preflight/<kind>-reference` and `<kind>-split`; stage writer locks under `checkpoints/<run_id>/`. The full registered scheduler budget is retained during the four-update preflight. All model, optimizer, scheduler, EMA, scaler-policy, RNG (Python/NumPy/CPU/all CUDA), sampler, mask/augmentation stream and deterministic metric state is compared. Tests certify exact recovery only on the same execution environment.

Exact resume: in a fresh `ahr-go` kernel load the Notebook 02 setup, reconstruct the identical configuration/protocol and bundle, and call the relevant function with the original artifact path. Source/config/environment changes require a new run ID. Completed compute cases and complete preflight models are reused only after identity/hash validation; an interrupted bounded synthetic compute case restarts its measurements. Training checkpoint manifests never silently restart after corruption. Production training longer than 30 minutes must use the main notebook's detached tmux workflow after this preflight passes.

Synthetic noise and changed token counts are **compute-only** inputs. They provide no evidence about Food-101 representation quality, image upscaling quality, or unmeasured asymptotic behavior. CPU allocator memory remains NOT_RUN. OOM, timeout and errors remain explicit, with no substituted timing or memory values.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


## Shared helpers
Definition loading only; all work requires an explicit function call.

In [ ]:
"""Explicit-only compute sweep and full-configuration checkpoint recovery checks."""
from pathlib import Path
import copy, datetime, gc, hashlib, json, math, os, platform, statistics, time
import numpy as np
import torch
from PIL import Image
from fmm_jepa.geometry import clear_geometry_cache
from fmm_jepa.training import atomic_json, definition_hashes, digest_file, execution_compatibility, move_masks, run_writer
from fmm_jepa.food101_training import (initialize_paired_models, run_model, validate_training_manifest,
    training_protocol, make_optimizer, pretraining_forward, isolated_rng, _rng, _restore_rng, _atomic_torch)
from fmm_jepa.food101_data import make_batch

FOOD_BENCHMARK_SCHEMA = 'fmm-jepa-food101-benchmark/v1'

def _food_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def _food_sync(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)

def _food_release(device):
    clear_geometry_cache()
    gc.collect()
    if torch.device(device).type == 'cuda':
        torch.cuda.empty_cache()
        _food_sync(device)

def _food_environment(config):
    value = execution_compatibility(config)
    value['cublas_workspace_config'] = os.environ.get('CUBLAS_WORKSPACE_CONFIG')
    value['hostname'] = platform.node()
    if str(config['device']).startswith('cuda'):
        props = torch.cuda.get_device_properties(config['device'])
        value['cuda_device_uuid'] = str(getattr(props, 'uuid', 'not_exposed'))
        value['cuda_device_capability'] = list(torch.cuda.get_device_capability(config['device']))
    return value

def _food_identity(root, config, extra=None):
    return {'schema': FOOD_BENCHMARK_SCHEMA, 'configuration': copy.deepcopy(config),
            'source_hashes': definition_hashes(root), 'environment': _food_environment(config),
            'extra': extra or {}}

def _food_immutable_manifest(path, identity):
    path = Path(path)
    if path.exists():
        manifest = json.loads(path.read_text())
        if manifest.get('identity') != identity:
            raise ValueError('Stage source/configuration/environment identity changed; use a new run ID')
        for entry in manifest.get('records', {}).values():
            record = Path(entry['path'])
            if digest_file(record) != entry['sha256']:
                raise ValueError('Stage record hash mismatch: ' + str(record))
        return manifest
    if path.parent.exists() and list(path.parent.glob('case_*.json')):
        raise ValueError('Orphan stage records without manifest; refusing silent replacement')
    manifest = {'identity': identity, 'identity_sha256': _food_hash(identity),
                'status': 'NOT_RUN', 'records': {}}
    atomic_json(path, manifest)
    return manifest

class SyntheticFoodComputeFixture:
    """Deterministic RGB noise only; never supplies representation-quality evidence."""
    def __init__(self, image_size, seed=302):
        self.image_size = int(image_size)
        self.seed = int(seed)
        self.fingerprint = _food_hash({'synthetic_rgb_noise': True, 'size': self.image_size, 'seed': self.seed})
    def get(self, identifier):
        seed = int(hashlib.sha256(f'{self.seed}:{identifier}'.encode()).hexdigest()[:16], 16)
        array = np.random.default_rng(seed).integers(0, 256, (self.image_size, self.image_size, 3), dtype=np.uint8)
        return {'image': Image.fromarray(array), 'label': 0, 'sample_id': identifier}

def _food_profile_call(call, device):
    # A real call in the actual gradient mode; full-JEPA includes predictor attention.
    activities = [torch.profiler.ProfilerActivity.CPU]
    if torch.device(device).type == 'cuda':
        activities.append(torch.profiler.ProfilerActivity.CUDA)
    with torch.profiler.profile(activities=activities) as profiler:
        call()
        _food_sync(device)
    events = sorted({entry.key for entry in profiler.key_averages()
                     if 'scaled_dot_product' in entry.key or 'attention' in entry.key.lower()})
    joined = ' '.join(events)
    if 'flash_attention_for_cpu' in joined:
        backend = 'pytorch_flash_attention_for_cpu'
    elif 'flash_attention' in joined:
        backend = 'pytorch_flash_attention_cuda'
    elif 'efficient_attention' in joined:
        backend = 'pytorch_memory_efficient_attention'
    elif 'cudnn_attention' in joined:
        backend = 'pytorch_cudnn_attention'
    elif 'attention_math' in joined:
        backend = 'pytorch_sdpa_math'
    else:
        backend = 'unclassified_attention_events' if events else 'no_attention_operator_observed'
    return {'backend': backend, 'events': events,
            'scope_note': 'FMM encoder has no attention; its full JEPA includes the common attention predictor.'}


## Matched compute sweep
Run the same initialized scaffold, inputs and masks sequentially for each architecture. Profile the actual attention dispatch outside timed repeats.

In [ ]:
def _food_benchmark_case(kind, scope, tokens, config, models, benchmark_policy):
    device = torch.device(config['device'])
    batch_size = int(benchmark_policy['microbatch'])
    grid_side = math.isqrt(tokens)
    if grid_side * grid_side != tokens:
        raise ValueError('The sweep requires square grids')
    local = copy.deepcopy(config)
    local['image_size'] = grid_side * int(config['model']['patch_size'])
    fixture = SyntheticFoodComputeFixture(local['image_size'], int(benchmark_policy['seed']))
    identifiers = [f'compute-only-{i:03d}' for i in range(batch_size)]
    cpu_batch = make_batch(fixture, identifiers, [0] * batch_size, local, training=True)
    tensor_hash = hashlib.sha256(cpu_batch['images'].contiguous().numpy().tobytes()).hexdigest()
    _food_release(device)
    preexisting = torch.cuda.memory_allocated(device) if device.type == 'cuda' else None
    model = optimizer = schedule = images = masks = None
    start = time.monotonic()
    try:
        images = cpu_batch['images'].to(device=device, dtype=torch.float32)
        masks = move_masks(cpu_batch['masks'], device)
        if scope == 'encoder_forward':
            model = copy.deepcopy(models[kind].target).to(device).eval().requires_grad_(False)
            def call():
                with torch.no_grad():
                    output = model(images)
                    if not bool(torch.isfinite(output).all()):
                        raise FloatingPointError('Nonfinite encoder output')
                    if output.shape != (batch_size, tokens, int(config['model']['dim'])):
                        raise AssertionError('Encoder token count/shape mismatch')
        elif scope == 'full_jepa_update':
            model = copy.deepcopy(models[kind]).to(device).train()
            # This compute sweep uses one microbatch per update, separately from the real accumulated preflight.
            policy = training_protocol(config)
            optimizer, schedule, _ = make_optimizer(model, policy)
            def call():
                optimizer.zero_grad(set_to_none=True)
                output = pretraining_forward(model, {'images': images, 'masks': masks})
                if not bool(torch.isfinite(output['loss'])):
                    raise FloatingPointError('Nonfinite JEPA loss')
                output['loss'].backward()
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],
                                              policy['gradient_clip'], error_if_nonfinite=True)
                if any(p.grad is not None for p in model.target.parameters()):
                    raise AssertionError('Teacher gradient leakage')
                optimizer.step()
                if not all(bool(torch.isfinite(p).all()) for p in model.parameters()):
                    raise FloatingPointError('Nonfinite updated JEPA parameters')
                model.update_teacher(schedule.ema_for(schedule.position + 1))
                schedule.step()
        else:
            raise ValueError('Unknown compute scope')
        for _ in range(benchmark_policy['warmup']):
            call()
            if time.monotonic() - start > benchmark_policy['case_timeout_seconds']:
                raise TimeoutError('Case time budget exceeded between complete calls')
        _food_sync(device)
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
        baseline = torch.cuda.memory_allocated(device) if device.type == 'cuda' else None
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        samples = []
        for _ in range(benchmark_policy['repeats']):
            _food_sync(device)
            tick = time.perf_counter()
            call()
            _food_sync(device)
            samples.append(time.perf_counter() - tick)
            if time.monotonic() - start > benchmark_policy['case_timeout_seconds']:
                raise TimeoutError('Case time budget exceeded between complete calls')
        peak = torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else None
        dispatch = _food_profile_call(call, device)
        if scope == 'encoder_forward' and kind == 'fmm' and dispatch['events']:
            raise AssertionError('Attention observed in the FMM encoder')
        if kind == 'vit' and not dispatch['events']:
            raise AssertionError('No attention dispatch was observed in ViT')
        median = statistics.median(samples)
        return {'status': 'PASS', 'model_id': kind, 'scope': scope, 'tokens': tokens,
                'image_size': local['image_size'], 'grid': [grid_side, grid_side],
                'microbatch': batch_size, 'accumulation_steps': 1, 'dtype': 'float32',
                'model_configuration': copy.deepcopy(config['model']),
                'fixture_fingerprint': fixture.fingerprint, 'image_tensor_sha256': tensor_hash,
                'mask_ids': cpu_batch['mask_ids'], 'augmentation_ids': cpu_batch['augmentation_ids'],
                'visible_tokens_mean': float(masks['context_valid'].sum(1).float().mean()),
                'target_tokens_mean': float(torch.cat([v.sum(1) for v in masks['target_valid']]).float().mean()),
                'warmup': benchmark_policy['warmup'], 'repeats': benchmark_policy['repeats'],
                'raw_seconds': samples, 'median_seconds': median,
                'mad_seconds': statistics.median(abs(x - median) for x in samples),
                'images_per_second': batch_size / median,
                'preexisting_cuda_allocated_bytes': preexisting,
                'baseline_allocated_bytes': baseline, 'peak_allocated_bytes': peak,
                'incremental_peak_bytes': peak - baseline if peak is not None else None,
                'memory_status': 'MEASURED' if device.type == 'cuda' else 'NOT_RUN_CPU_ALLOCATOR',
                'memory_definition': 'CUDA max_memory_allocated over measured calls after warmup. Total includes model, optimizer, inputs, cached geometry and preexisting allocations; incremental subtracts baseline. Reserved allocator/context excluded.',
                'dispatch': dispatch,
                'timing_scope': 'Synchronized calls; excludes data/preprocessing/transfers, checkpoint I/O, warmup and profiler. Encoder includes finite/shape checks; full JEPA includes loss, backward, clipping, optimizer, scheduler and EMA with one microbatch.',
                'quality_evidence': False}
    finally:
        model = optimizer = schedule = images = masks = None
        _food_release(device)


def benchmark_compute(root, artifact, config):
    """Explicit synthetic-only matched sweep; never called on import or authoring."""
    root, artifact = Path(root), Path(artifact)
    if config.get('dtype', 'float32') != 'float32':
        raise ValueError('The registered sweep supports FP32 only')
    expected = {'patch_size': 8, 'dim': 192, 'depth': 4, 'order': 4, 'leaf_size': 2}
    if any(config['model'].get(k) != v for k, v in expected.items()):
        raise ValueError('Model scaffold differs from the registered compute sweep')
    policy = {'token_counts': [64, 256, 1024], 'microbatch': 2, 'warmup': 2,
              'repeats': 3, 'seed': 20260925, 'case_timeout_seconds': 900}
    policy.update(config.get('benchmark', {}))
    if policy['token_counts'] != [64, 256, 1024] or policy['microbatch'] < 1:
        raise ValueError('Expected all three token counts and a positive microbatch')
    if policy['warmup'] < 2 or policy['repeats'] < 3:
        raise ValueError('Require at least two warmup and three measured calls')
    directory = artifact / 'compute_benchmark'
    directory.mkdir(parents=True, exist_ok=True)
    manifest_path = directory / 'manifest.json'
    identity = _food_identity(root, config, {'stage': 'compute-only-sweep', 'policy': policy})
    lock_dir = root / 'checkpoints' / artifact.name / 'compute_benchmark'
    with run_writer(lock_dir), isolated_rng(policy['seed']):
        manifest = _food_immutable_manifest(manifest_path, identity)
        models, counts, initialization = initialize_paired_models(config, directory)
        rows = []
        for tokens in policy['token_counts']:
            for scope in ('encoder_forward', 'full_jepa_update'):
                for kind in ('vit', 'fmm'):
                    key = f'{tokens}_{scope}_{kind}'
                    path = directory / f'case_{key}.json'
                    if key in manifest['records']:
                        row = json.loads(path.read_text())
                    else:
                        if path.exists():
                            # A completed record may precede its manifest publication after interruption.
                            row = json.loads(path.read_text())
                            if row.get('identity_sha256') != manifest['identity_sha256'] or row.get('case_key') != key:
                                raise ValueError('Orphan case record identity mismatch')
                        else:
                            manifest.update(status='RUNNING', current_case=key)
                            atomic_json(manifest_path, manifest)
                            try:
                                row = _food_benchmark_case(kind, scope, tokens, config, models, policy)
                            except (RuntimeError, FloatingPointError, TimeoutError, AssertionError) as exc:
                                oom = isinstance(exc, torch.cuda.OutOfMemoryError) or 'out of memory' in str(exc).lower()
                                row = {'status': 'OOM' if oom else 'TIMEOUT' if isinstance(exc, TimeoutError) else 'ERROR',
                                       'model_id': kind, 'scope': scope, 'tokens': tokens, 'error': repr(exc),
                                       'median_seconds': None, 'peak_allocated_bytes': None, 'quality_evidence': False}
                            row.update(identity_sha256=manifest['identity_sha256'], case_key=key)
                            atomic_json(path, row)
                        manifest['records'][key] = {'path': str(path.resolve()), 'sha256': digest_file(path)}
                        atomic_json(manifest_path, manifest)
                    rows.append(row)
                    print(f"Compute {key}: {row['status']}", flush=True)
        for tokens in policy['token_counts']:
            for scope in ('encoder_forward', 'full_jepa_update'):
                pair = [r for r in rows if r['tokens'] == tokens and r['scope'] == scope and r['status'] == 'PASS']
                if len(pair) == 2:
                    for field in ('image_tensor_sha256', 'mask_ids', 'augmentation_ids'):
                        if pair[0][field] != pair[1][field]:
                            raise AssertionError('Paired fixture or mask mismatch')
        report = {'schema': FOOD_BENCHMARK_SCHEMA,
                  'status': 'PASS' if all(r['status'] == 'PASS' for r in rows) else 'INCOMPLETE',
                  'identity': identity, 'identity_sha256': manifest['identity_sha256'],
                  'policy': policy, 'rows': rows, 'parameter_counts': counts,
                  'quality_evidence': False,
                  'interpretation': 'Synthetic RGB compute only. These timing and allocation measurements do not establish useful learning, natural-image generalization, or efficiency at unmeasured token counts.',
                  'resume': 'Call benchmark_compute with the identical root/artifact/config/environment. Validated completed cases are reused; only an interrupted bounded measurement case restarts. OOM/ERROR/TIMEOUT remain explicit terminal measurements; a changed policy requires a new run ID.'}
        atomic_json(artifact / 'compute_benchmark.json', report)
        manifest.update(status=report['status'], current_case=None,
                        summary_sha256=digest_file(artifact / 'compute_benchmark.json'))
        atomic_json(manifest_path, manifest)
        return report


## Actual-configuration recovery preflight
Four uninterrupted updates must equal two updates followed by reconstruction and two resumed updates. Timing differences are excluded from state equality.

In [ ]:
def _food_exact_tree(left, right, path='root'):
    if isinstance(left, np.ndarray):
        if not isinstance(right, np.ndarray) or left.dtype != right.dtype or not np.array_equal(left, right):
            raise AssertionError('NumPy state differs: ' + path)
    elif torch.is_tensor(left):
        if not torch.is_tensor(right) or left.dtype != right.dtype or not torch.equal(left, right):
            raise AssertionError('Tensor state differs: ' + path)
    elif isinstance(left, dict):
        if not isinstance(right, dict) or left.keys() != right.keys():
            raise AssertionError('State keys differ: ' + path)
        for key in left:
            _food_exact_tree(left[key], right[key], path + '.' + str(key))
    elif isinstance(left, (list, tuple)):
        if type(left) != type(right) or len(left) != len(right):
            raise AssertionError('State sequence differs: ' + path)
        for index, (a, b) in enumerate(zip(left, right)):
            _food_exact_tree(a, b, path + f'[{index}]')
    elif left != right:
        raise AssertionError('State value differs: ' + path)


def _food_preflight_payload(directory):
    manifest = validate_training_manifest(directory)
    if len(manifest['checkpoints']) < 2:
        raise AssertionError('Latest two valid resumable checkpoints were not retained')
    value = torch.load(Path(directory) / manifest['latest'], map_location='cpu', weights_only=False)
    return manifest, value


def verify_food_resume(root, artifact, config, bundle, protocol):
    """Actual-config K4/J2 uninterrupted vs checkpoint-reconstructed trajectories, both models."""
    from fmm_jepa.food101_data import EvaluationDataset
    from fmm_jepa.food101_evaluation import compute_diagnostics
    root, artifact = Path(root), Path(artifact)
    diagnostic_ids = list(bundle['diagnostic_ids'])[:8]
    if len(diagnostic_ids) < 2 or not set(diagnostic_ids).issubset(bundle['dev_ids']):
        raise ValueError('Preflight requires at least two fixed development diagnostic examples')
    diagnostic_data = EvaluationDataset(bundle['dataset'], diagnostic_ids, config)
    def diagnostic_callback(model, kind, step):
        value = compute_diagnostics(model.target, diagnostic_data, config)
        value.update(model_id=kind, step=step, purpose='preflight evaluation-cache-to-backward regression; not quality evidence')
        return value
    policy = training_protocol(config)
    if policy['updates'] != 5000 or policy['effective_batch'] != 128:
        raise ValueError('The registered Food pilot uses 5000 successful updates and effective batch128')
    if config.get('evaluation', {}).get('heldout_enabled', False):
        raise ValueError('Held-out prediction must remain disabled')
    if not torch.are_deterministic_algorithms_enabled():
        raise ValueError('Enable deterministic algorithms before the exact-recovery preflight')
    if str(config['device']).startswith('cuda') and os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
        raise ValueError('Set CUBLAS_WORKSPACE_CONFIG=:4096:8 before importing/initializing CUDA')
    directory = artifact / 'preflight'
    directory.mkdir(parents=True, exist_ok=True)
    checkpoint_root = root / 'checkpoints' / artifact.name / 'preflight'
    identity = _food_identity(root, config, {'stage': 'actual-config-resume', 'K': 4, 'J': 2,
        'protocol': protocol, 'diagnostic_ids': diagnostic_ids, 'dataset_fingerprint': bundle['dataset_manifest']['dataset_fingerprint'],
        'split_sha256': bundle['split_manifest']['split_sha256']})
    manifest_path = directory / 'manifest.json'
    original_rng = _rng()
    results = {}
    with run_writer(checkpoint_root):
        manifest = _food_immutable_manifest(manifest_path, identity)
        models, counts, initialization = initialize_paired_models(config, directory)
        rng_path = directory / 'trajectory_start_rng.pt'
        if 'trajectory_start_rng' in manifest:
            if digest_file(rng_path) != manifest['trajectory_start_rng']['sha256']:
                raise ValueError('Preflight starting RNG state hash mismatch')
            trajectory_rng = torch.load(rng_path, map_location='cpu', weights_only=False)
        else:
            if rng_path.exists():
                raise ValueError('Orphan preflight starting RNG file; preserve evidence and use a new run ID')
            trajectory_rng = copy.deepcopy(original_rng)
            _atomic_torch(rng_path, trajectory_rng)
            manifest['trajectory_start_rng'] = {'path': str(rng_path.resolve()), 'sha256': digest_file(rng_path)}
            atomic_json(manifest_path, manifest)
        # Independent isolated preflight streams; production model controls are never updated here.
        preflight_protocol = copy.deepcopy(protocol)
        preflight_protocol['preflight'] = {'K': 4, 'J': 2, 'schedule_budget_unchanged': 5000,
                                         'stage_identity_sha256': manifest['identity_sha256'],
                                         'diagnostic_ids': diagnostic_ids,
                                         'diagnostic_purpose': 'evaluation-cache-to-backward regression at the actual image size'}
        try:
            for kind in ('vit', 'fmm'):
                record_path = directory / f'case_{kind}.json'
                if kind in manifest['records']:
                    record = json.loads(record_path.read_text())
                    for name, hash_key in (('reference_directory', 'reference_latest_sha256'),
                                           ('resumed_directory', 'resumed_latest_sha256')):
                        checked = validate_training_manifest(record[name])
                        if checked['successful_updates'] != 4 or digest_file(Path(record[name]) / checked['latest']) != record[hash_key]:
                            raise ValueError('Completed preflight checkpoint has changed')
                    results[kind] = record
                    continue
                print(f"{kind}: actual image{config['image_size']}, microbatch{policy['microbatch']} x accumulation{policy['accumulation_steps']}; reference4 vs restored2+2", flush=True)
                reference_dir, split_dir = checkpoint_root / f'{kind}-reference', checkpoint_root / f'{kind}-split'
                _restore_rng(copy.deepcopy(trajectory_rng))
                reference = run_model(kind, config, bundle, directory / f'{kind}-reference', reference_dir,
                                      preflight_protocol, model=copy.deepcopy(models[kind]), stop_after=4, diagnostic_callback=diagnostic_callback)
                reference_manifest, reference_payload = _food_preflight_payload(reference_dir)
                performance = copy.deepcopy(reference['performance'])
                del reference
                _food_release(config['device'])
                split_marker = directory / f'{kind}_split_boundary.json'
                split_manifest = validate_training_manifest(split_dir) if (split_dir / 'manifest.json').exists() else None
                if split_manifest is None or split_manifest['successful_updates'] < 2:
                    _restore_rng(copy.deepcopy(trajectory_rng))
                    partial = run_model(kind, config, bundle, directory / f'{kind}-split', split_dir,
                                        preflight_protocol, model=copy.deepcopy(models[kind]), stop_after=2, diagnostic_callback=diagnostic_callback)
                    if partial['resume']['final_step'] != 2:
                        raise AssertionError('The split boundary was not reached exactly')
                    del partial
                    _food_release(config['device'])
                    split_manifest = validate_training_manifest(split_dir)
                if not split_marker.exists():
                    if split_manifest['successful_updates'] != 2:
                        raise ValueError('Missing split-boundary evidence; refusing to infer an unobserved resume')
                    atomic_json(split_marker, {'identity_sha256': manifest['identity_sha256'], 'kind': kind,
                        'successful_updates': 2, 'file': split_manifest['latest'],
                        'sha256': digest_file(split_dir / split_manifest['latest'])})
                marker = json.loads(split_marker.read_text())
                if marker['identity_sha256'] != manifest['identity_sha256'] or marker['successful_updates'] != 2:
                    raise ValueError('Preflight split marker mismatch')
                resumed = run_model(kind, config, bundle, directory / f'{kind}-split', split_dir,
                                    preflight_protocol, model=copy.deepcopy(models[kind]), stop_after=4, diagnostic_callback=diagnostic_callback)
                if resumed['resume']['resumed_from_step'] not in (2, 4):
                    raise AssertionError('Expected recovery from step2 or validated completed step4')
                continued_manifest, continued_payload = _food_preflight_payload(split_dir)
                # Full state equality except nondeterministic timers and invocation labels.
                fields = ['model', 'optimizer', 'scheduler', 'ema_position', 'scaler', 'precision', 'rng',
                          'sampler', 'streams', 'microstep', 'partial_accumulation', 'world_size', 'rank',
                          'replay', 'config', 'protocol', 'identity']
                for field in fields:
                    _food_exact_tree(reference_payload[field], continued_payload[field], field)
                state_fields = ['successful_updates', 'attempted_updates', 'image_exposures',
                                'consecutive_failures', 'traces', 'failures', 'diagnostics']
                for field in state_fields:
                    _food_exact_tree(reference_payload['state'][field], continued_payload['state'][field], 'state.' + field)
                timing_fields = {'data_seconds', 'compute_seconds', 'step_seconds', 'images_per_second', '_cuda_event_count'}
                def deterministic_metrics(payload):
                    return [{k: v for k, v in row.items() if k not in timing_fields} for row in payload['state']['metrics']]
                _food_exact_tree(deterministic_metrics(reference_payload), deterministic_metrics(continued_payload), 'deterministic_metrics')
                exposures = 4 * policy['effective_batch']
                if reference_payload['state']['successful_updates'] != 4 or reference_payload['state']['attempted_updates'] != 4:
                    raise AssertionError('Unexpected failed/extra attempt in recovery smoke')
                if reference_payload['state']['image_exposures'] != exposures or reference_payload['sampler']['cursor'] != exposures:
                    raise AssertionError('Exposure/sampler counters differ from actual full batches')
                # Step2 must still be retained alongside step4 (plus best) for this four-step trajectory.
                if digest_file(split_dir / marker['file']) != marker['sha256']:
                    raise ValueError('Original split checkpoint missing or changed')
                record = {'status': 'PASS', 'bitwise_equal': True, 'model_id': kind,
                    'identity_sha256': manifest['identity_sha256'], 'successful_updates': 4, 'split_at': 2,
                    'attempted_updates': 4, 'image_exposures': exposures, 'latest_two_valid': True,
                    'diagnostic_to_training_transition_verified': True, 'diagnostic_ids': diagnostic_ids,
                    'compared_fields': fields + ['state.' + key for key in state_fields] + ['all metrics except elapsed-time/throughput fields'],
                    'microbatch': policy['microbatch'], 'accumulation_steps': policy['accumulation_steps'],
                    'schedule_budget': policy['updates'], 'image_size': config['image_size'],
                    'device': config['device'], 'dtype': config.get('dtype', 'float32'),
                    'reference_directory': str(reference_dir.resolve()), 'resumed_directory': str(split_dir.resolve()),
                    'reference_latest_sha256': digest_file(reference_dir / reference_manifest['latest']),
                    'resumed_latest_sha256': digest_file(split_dir / continued_manifest['latest']),
                    'performance': performance, 'scope': 'Same hardware/software/configuration; does not certify cross-device or cross-version equality.'}
                atomic_json(record_path, record)
                manifest['records'][kind] = {'path': str(record_path.resolve()), 'sha256': digest_file(record_path)}
                manifest.update(status='RUNNING', current_model=kind)
                atomic_json(manifest_path, manifest)
                results[kind] = record
                del reference_payload, continued_payload, resumed
                _food_release(config['device'])
        except BaseException as exc:
            manifest.update(status='FAILED', error=repr(exc))
            atomic_json(manifest_path, manifest)
            raise
        finally:
            _restore_rng(original_rng)
            _food_release(config['device'])
        estimates = {}
        for kind, record in results.items():
            measured = record['performance'].get('complete_jepa_step_seconds')
            estimates[kind] = 5000 * measured['median'] if measured else None
        total = sum(estimates.values()) if all(v is not None for v in estimates.values()) else None
        report = {'schema': FOOD_BENCHMARK_SCHEMA, 'status': 'PASS', 'identity': identity,
                  'identity_sha256': manifest['identity_sha256'], 'models': results,
                  'parameter_counts': counts, 'expected_training_seconds': estimates,
                  'estimated_pair_training_hours': total / 3600 if total is not None else None,
                  'estimated_pair_with_overhead_hours': total * 1.25 / 3600 if total is not None else None,
                  'estimate_limit': 'Only two post-warmup measured updates per model. Includes actual accumulated JEPA and data preparation; excludes later frozen-feature/probe evaluation and checkpoint/diagnostic overhead. 25% allowance is not a guarantee.',
                  'long_training_requires_detached_tmux': True, 'heldout_enabled': False,
                  'source_hashes': identity['source_hashes'],
                  'resume': 'Recreate the same configuration/protocol/bundle in a fresh ahr-go kernel and call verify_food_resume with the same root and artifact. Retained manifests and stage hashes are validated before reuse; invalid files never cause silent restart.'}
        atomic_json(artifact / 'preflight_certificate.json', report)
        manifest.update(status='PASS', current_model=None,
                        certificate_sha256=digest_file(artifact / 'preflight_certificate.json'))
        atomic_json(manifest_path, manifest)
        return report
